## Load your data (for public sharing)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import gdown
import pandas as pd
import zipfile # Import zipfile
import os # Import os for path checking

# Upload the data file to your Google Drive, and turn on the link sharing
# Replace 'YOUR_FILE_ID' with the actual file ID from your public Google Drive link e.g., "https://drive.google.com/file/d/YOUR_FILE_ID/view?usp=sharing"
# Example data is from https://www.kaggle.com/datasets/carolzhangdc/imdb-5000-movie-dataset/data
file_id = '/content/drive/MyDrive/KoBBQ-main.zip'

# Name for the downloaded file in Colab
# Since `file_id` now points to a local .zip file in Google Drive,
# we will unzip it. Let's assume the zip file contains a CSV named 'KoBBQ-main.csv'.
# We'll use this name for loading the DataFrame.
output_filename = 'KoBBQ-main/data/KoBBQ_all_samples.tsv' # Corrected path to the extracted TSV file.

# The gdown.download function failed because `id` parameter expects a Google Drive file ID
# (e.g., '1AbCdEfGhIjKlMnOpQrStUvWxYz') for a publicly shared file, but it received a local file path.
# Since the file is already mounted via Google Drive, gdown is not needed here.
# Instead, we need to unzip the local .zip file and then read the CSV.
# gdown.download(id=file_id, output=output_filename, quiet=False) # This line caused the error

# Unzip the file
try:
    with zipfile.ZipFile(file_id, 'r') as zip_ref:
        zip_ref.extractall('.') # Extract contents to the current directory
    print(f"File '{file_id}' unzipped successfully!")

    # Verify that the expected CSV file exists after extraction
    if not os.path.exists(output_filename):
        print(f"Warning: Expected TSV '{output_filename}' not found after unzipping '{file_id}'.")
        print("Please check the actual name of the TSV file inside the zip archive and its path.")
        # Optional: list files in current directory to help user identify correct name
        print(f"Files in current directory: {os.listdir('.')}")
        # If the file doesn't exist, reading it will fail.
        # Consider adding an exit() or raising an error here if crucial.

except zipfile.BadZipFile:
    print(f"Error: '{file_id}' is not a valid zip file. Please check the file.")
    exit() # Stop execution if the zip file is invalid.
except FileNotFoundError:
    print(f"Error: Zip file not found at '{file_id}'. Ensure it's in your Google Drive and mounted correctly.")
    exit() # Stop execution if the zip file is not found.

# Now you can load it with pandas, specifying the tab separator
df_raw = pd.read_csv(output_filename, sep='\t')
display(df_raw.head())

In [ ]:
# ── Q1 의존 패키지 설치 ──────────────────────────────────────────
!pip install konlpy kiwipiepy transformers -q

import re
import pandas as pd

# ── 예시 샘플 추출 ────────────────────────────────────────────────
# 각 bbq_category × {amb, dis} 조합에서 1행씩 추출 → context + question 쌍
samples = []

for cat in df_raw['bbq_category'].dropna().unique():
    for cond in ['amb', 'dis']:
        mask = (
            df_raw['bbq_category'] == cat
        ) & (
            df_raw['sample_id'].str.contains(f'-{cond}-')
        )
        subset = df_raw[mask]
        if len(subset) == 0:
            continue
        row = subset.iloc[0]
        # context 행
        samples.append({
            'category': cat,
            'condition': cond,
            'field': 'context',
            'text': row['context']
        })
        # question 행
        samples.append({
            'category': cat,
            'condition': cond,
            'field': 'question',
            'text': row['question']
        })

df_samples = pd.DataFrame(samples)
# 약 20개 안팎으로 제한
df_samples = df_samples.head(20).reset_index(drop=True)

print(f"샘플 수: {len(df_samples)}")
display(df_samples[['category', 'condition', 'field', 'text']])

## Q1: Text cleaning

So far we've learned a few text cleaning perations, let's put them together in a function! This function would be a handy one to refer to if you happen to work with some messy text data, and you want to preprocess it with a single function.

Write a function that has at least the following:
- Lowercase the text
- Remove punctuation marks
- Remove extra whitespace characters

In [ ]:
# ── clean_text 함수 ───────────────────────────────────────────────
def clean_text(text: str) -> str:
    """
    한국어 텍스트 정제 함수
    1. 불필요한 HTML 태그 제거 (혹시 포함될 경우)
    2. 특수문자·구두점 제거 (한글, 영문, 숫자, 공백만 유지)
    3. 소문자 변환 (영문 포함 대비)
    4. 연속 공백 → 단일 공백으로 정규화
    5. 앞뒤 공백 제거
    """
    if not isinstance(text, str):
        return ''

    # Step 1: HTML 태그 제거
    text = re.sub(r'<[^>]+>', '', text)

    # Step 2: 구두점·특수문자 제거 (한글 \uAC00-\uD7A3, 영문, 숫자, 공백 유지)
    text = re.sub(r'[^\uAC00-\uD7A3a-zA-Z0-9\s]', '', text)

    # Step 3: 소문자 변환 (영문 대비)
    text = text.lower()

    # Step 4: 연속 공백 → 단일 공백
    text = re.sub(r'\s+', ' ', text)

    # Step 5: 앞뒤 공백 제거
    text = text.strip()

    return text

In [ ]:
# Apply the cleaning function to your text column of the DataFrame
df_samples['cleaned_text'] = df_samples['text'].apply(clean_text)

display(df_samples[['text', 'cleaned_text']].head(20))

## Q2: Should We Remove Stop Words?

Stop-word removal is useful in some NLP tasks, but it can also remove meaningful information.  
In this question, you will decide whether stop-word removal is appropriate for your own dataset.

Choose 10–20 examples from your text column. Then apply tokenization and compare the text before and after stop-word removal.

For English data, use `nltk` or `spaCy` stop-word lists.  
For Korean data, use a custom stop-word list or a morphological analyzer such as `Okt` or `Kiwi`.

Answer the following questions:

1. Remove stop words using one or more methods taught in class.
2. Observe which words were removed. Did stop-word removal help your analysis, or did it remove meaningful information?
3. Writhe whether your would remove stop words in your final analysis? Why or why not?

Your answer should include code, output examples, and a short written justification.

In [ ]:
# Your code that removes stopwords from the cleaned text and prints out the results

In [ ]:
# ── Q2: 불용어 제거 ───────────────────────────────────────────────
from konlpy.tag import Okt

okt = Okt()

# 한국어 불용어 리스트 (일반적으로 많이 쓰이는 조사·어미·접속사 등)
korean_stopwords = [
    '이', '가', '은', '는', '을', '를', '의', '에', '에서', '로', '으로',
    '와', '과', '도', '만', '까지', '부터', '에게', '한테', '께',
    '이다', '있다', '없다', '하다', '되다', '것', '수', '등', '및',
    '그', '그리고', '그러나', '하지만', '또는', '또', '즉', '때문에',
    '위해', '통해', '대해', '관해', '대한', '위한', '인해',
    '들', '들이', '들은', '들을', '들의', '들에',
    '했습니다', '합니다', '입니다', '습니다', '니다',
    '어', '아', '야', '요', '죠', '네', '나', '지'
]

def remove_stopwords(text: str, stopwords: list) -> str:
    """
    Okt 형태소 분석기로 토크나이징 후 불용어 제거
    Returns: 불용어 제거된 어절을 공백으로 이어 붙인 문자열
    """
    if not isinstance(text, str) or text.strip() == '':
        return ''
    # morphs: 형태소 단위 토큰 리스트
    tokens = okt.morphs(text, stem=True)  # stem=True: 어간 추출
    filtered = [tok for tok in tokens if tok not in stopwords]
    return ' '.join(filtered)

# ── 적용 ─────────────────────────────────────────────────────────
df_samples['tokens_okt'] = df_samples['cleaned_text'].apply(
    lambda x: okt.morphs(x, stem=True)
)
df_samples['tokens_no_stopwords'] = df_samples['cleaned_text'].apply(
    lambda x: remove_stopwords(x, korean_stopwords)
)

In [ ]:
# Compare the results of before and after stopword removal

In [ ]:
# ── 비교 출력 ────────────────────────────────────────────────────
print("=== 불용어 제거 전후 비교 (10개 예시) ===\n")
for i, row in df_samples.head(10).iterrows():
    print(f"[{i}] category={row['category']} | condition={row['condition']} | field={row['field']}")
    print(f"  원문        : {row['text']}")
    print(f"  정제 후     : {row['cleaned_text']}")
    print(f"  토큰(Okt)   : {row['tokens_okt']}")
    print(f"  불용어 제거 : {row['tokens_no_stopwords']}")
    print()

In [ ]:
# ── 제거된 불용어 확인 ────────────────────────────────────────────
print("=== 제거된 불용어 예시 ===\n")
for i, row in df_samples.head(10).iterrows():
    original_tokens = row['tokens_okt']
    filtered_tokens = row['tokens_no_stopwords'].split()
    removed = [t for t in original_tokens if t not in filtered_tokens]
    print(f"[{i}] {row['field']} | 제거된 토큰: {removed}")

Your answer whether you would remove stop words in your final analysis? Why or why not?
**Answer:**

감정 분석을 진행할 계획이 있어 불용어 제거는 신중하게 접근해야 한다. 특히 예시 5번과 7번과 같이 부정 표현 등이 불용어로 분류되어 제거될 경우, 감정의 극성(긍정/부정)이 왜곡되거나 강도가 약화될 수 있다. 불용어를 제거하는 편이 노이즈나 계산 효율성을 고려했을 때 용이다고 생각한다. 다만 앞서 언급한 문제점이 발생할 가능성도 있기 때문에 감정 분석에 중요한 영향을 미칠 수 있는 단어들은 불용어 리스트에서 제외하는 과정이 추가로 필요하다.

## Q3: Choose a Tokenizer for Your Own Text Data

Tokenization is one of the most important steps in text preprocessing. However, there is no single tokenizer that works best for every dataset. The best choice depends on the language, text source, and analysis goal.

Choose 10–20 examples from your own text column and inspect them carefully. Then compare different tokenization methods.

Answer the following questions:

1. What is the language of your data? If your data contains Korean, why might simple whitespace tokenization be insufficient?

2. What kind of text is it? For example: survey responses, social media posts, interviews, reviews, news articles, academic abstracts, or mixed sources.

3. Apply at least two tokenization methods to the same 5–10 text examples.  
   - For English data, you may compare methods such as:
     - whitespace tokenization
     - NLTK `word_tokenize`
     - spaCy tokenizer
     - a BERT tokenizer
   - For Korean data, you may compare methods such as:
     - whitespace tokenization
     - `Okt`
     - `Kiwi`
     - `Komoran`
     - `Kkma`
     - a Korean BERT tokenizer

4. Compare the outputs. What differences do you notice? Consider:
   - Are words split in a useful way?
   - Are particles, endings, punctuation, or symbols handled appropriately?
   - Are unknown words, slang, abbreviations, or domain-specific terms handled well?
   - Does the tokenizer preserve information that may be important for your analysis?

5. Which tokenizer would you choose for your dataset, and why?

Your answer should include:
- code that applies at least two tokenizers,
- tokenized output for several examples,
- a short written justification for your final tokenizer choice.

In [ ]:
# Your code that compares the results of the two or more methods and prints them out

In [ ]:
# ── Q3 추가 설치 ─────────────────────────────────────────────────
!pip install kiwipiepy transformers -q

from konlpy.tag import Okt
from kiwipiepy import Kiwi
from transformers import AutoTokenizer

okt   = Okt()
kiwi  = Kiwi()
bert_tokenizer = AutoTokenizer.from_pretrained('snunlp/KR-ELECTRA-discriminator')
# ※ KoBERT 대신 공개된 한국어 BERT 계열인 KR-ELECTRA 사용
#   (monologg/kobert 는 sentencepiece 별도 설치 필요)

# ── 세 토크나이저 적용 함수 ───────────────────────────────────────
def tokenize_okt(text: str) -> list:
    """Okt 형태소 분석 (어간 추출 포함)"""
    return okt.morphs(text, stem=True)

def tokenize_kiwi(text: str) -> list:
    """Kiwi 형태소 분석 → 형태소 문자열 리스트"""
    result = kiwi.analyze(text)
    # analyze() 결과: [(Token, ...), score] 구조
    tokens = [token.form for token in result[0][0]]
    return tokens

def tokenize_bert(text: str) -> list:
    """KR-ELECTRA (WordPiece) 서브워드 토크나이저"""
    return bert_tokenizer.tokenize(text)

# ── 전체 샘플에 적용 ─────────────────────────────────────────────
df_samples['tok_okt']  = df_samples['cleaned_text'].apply(tokenize_okt)
df_samples['tok_kiwi'] = df_samples['cleaned_text'].apply(tokenize_kiwi)
df_samples['tok_bert'] = df_samples['cleaned_text'].apply(tokenize_bert)

# ── 비교 출력 ────────────────────────────────────────────────────
print("=== 토크나이저 비교 (20개 샘플) ===\n")
for i, row in df_samples.iterrows():
    print(f"[{i:02d}] category={row['category']} | condition={row['condition']} | field={row['field']}")
    print(f"  원문   : {row['text']}")
    print(f"  Okt    : {row['tok_okt']}")
    print(f"  Kiwi   : {row['tok_kiwi']}")
    print(f"  KoBERT : {row['tok_bert']}")
    print()

In [ ]:
# ── 토큰 수 비교 요약 테이블 ─────────────────────────────────────
df_samples['n_okt']  = df_samples['tok_okt'].apply(len)
df_samples['n_kiwi'] = df_samples['tok_kiwi'].apply(len)
df_samples['n_bert'] = df_samples['tok_bert'].apply(len)

summary = df_samples[['category', 'condition', 'field',
                       'n_okt', 'n_kiwi', 'n_bert']]
print("=== 토큰 수 비교 ===")
display(summary)

print("\n=== 평균 토큰 수 ===")
print(summary[['n_okt', 'n_kiwi', 'n_bert']].mean().round(2))

Your answer which tokenizer you think is better for your data and task

**Answer:**

okt와 kiwi가 bert보다 한국어 토크나이저로써 더 좋은 성능을 보이고 있다고 생각한다. bert는 핵심 명사를 의미 경계와 무관하게 분절하여 QA 태스크의
답 후보 단어를 온전히 보존하지 못한다.

okt와 kiwi 둘 중에서는 okt가 KoBBQ 데이터에 더 적합하다고 생각한다. kiwi는 okt보다 형태소를 세분화해서 보여주고 있어 평균 토큰 수가 더 많은 것으로 나타난다. KoBBQ는 간결한 문장이기에 kiwi를 이용할 경우 분석 단위가 지나치게 작아진다.

# 4. What else? (bonus point)

각 형태소에 품사를 할당하는 Parts Of Speech(POS) Tagging 활용

KoBBQ에서 POS Tagging이 의미 있는 이유:

*   context의 핵심 인물 명사(NNP, NNG) 추출 → 편향 분석 대상 파악
*   question의 의문사/서술어 패턴 파악
*   불필요한 조사(JX, JKS 등)를 품사 기준으로 정밀 제거 가능

(참고: https://www.geeksforgeeks.org/nlp/text-preprocessing-for-nlp-tasks/)

In [ ]:
# ── 4단계: POS Tagging ─────────────────────────────────────────────
# Okt를 최종 토크나이저로 선택했으므로 Okt POS 태깅 사용
# Okt 주요 품사 태그:
#   Noun(명사), Verb(동사), Adjective(형용사), Adverb(부사)
#   Josa(조사), Eomi(어미), Punctuation(구두점)
#   Alpha(영문자), Number(숫자), Foreign(외국어)

from konlpy.tag import Okt
import pandas as pd

okt = Okt()

# ── POS 태깅 함수 ──────────────────────────────────────────────────
def pos_tag(text: str) -> list:
    """
    Okt POS 태거
    Returns: [(형태소, 품사), ...] 리스트
    """
    if not isinstance(text, str) or text.strip() == '':
        return []
    return okt.pos(text, stem=True, norm=True)
    # stem=True : 어간 추출 (먹었다 → 먹다)
    # norm=True : 정규화 (ㅋㅋ → 크크)

def extract_by_pos(text: str, target_pos: list) -> list:
    """
    특정 품사만 필터링해서 반환
    예: target_pos=['Noun'] → 명사만 추출
    """
    tagged = pos_tag(text)
    return [word for word, pos in tagged if pos in target_pos]

# ── 전체 샘플에 적용 ───────────────────────────────────────────────
df_samples['pos_tags']  = df_samples['cleaned_text'].apply(pos_tag)
df_samples['nouns']     = df_samples['cleaned_text'].apply(
    lambda x: extract_by_pos(x, ['Noun'])
)
df_samples['verbs']     = df_samples['cleaned_text'].apply(
    lambda x: extract_by_pos(x, ['Verb', 'Adjective'])
)
df_samples['josa']      = df_samples['cleaned_text'].apply(
    lambda x: extract_by_pos(x, ['Josa'])
)

# ── 전체 결과 출력 ─────────────────────────────────────────────────
print("=== POS Tagging 전체 결과 ===\n")
for i, row in df_samples.iterrows():
    print(f"[{i:02d}] category={row['category']} | "
          f"condition={row['condition']} | field={row['field']}")
    print(f"  원문        : {row['text']}")
    print(f"  POS 태그    : {row['pos_tags']}")
    print(f"  명사        : {row['nouns']}")
    print(f"  동사/형용사 : {row['verbs']}")
    print(f"  조사        : {row['josa']}")
    print()

In [ ]:
# ── KoBBQ 특화 분석: bbq_category별 핵심 명사 분포 ─────────────────
# KoBBQ는 편향(bias) 탐지 데이터셋이므로
# 어떤 카테고리에 어떤 명사가 등장하는지 파악하면
# 편향 분석에 활용 가능

from collections import Counter

print("=== bbq_category × condition 별 핵심 명사 Top 5 ===\n")

for cat in df_samples['category'].unique():
    for cond in ['amb', 'dis']:
        subset = df_samples[
            (df_samples['category'] == cat) &
            (df_samples['condition'] == cond)
        ]
        if subset.empty:
            continue

        # context + question 명사를 합쳐서 빈도 계산
        all_nouns = []
        for nouns in subset['nouns']:
            all_nouns.extend(nouns)

        top5 = Counter(all_nouns).most_common(5)
        print(f"  [{cat} | {cond}] 상위 명사: {top5}")
print()

In [ ]:
# ── context vs question POS 분포 비교 ─────────────────────────────
# context와 question이 어떤 품사 구성을 갖는지 비교

def pos_distribution(pos_tag_list: list) -> dict:
    """품사 태그 리스트 → 품사별 비율 딕셔너리"""
    counter = Counter(pos for _, pos in pos_tag_list)
    total = sum(counter.values())
    if total == 0:
        return {}
    return {pos: round(cnt / total * 100, 1)
            for pos, cnt in counter.most_common()}

df_samples['pos_dist'] = df_samples['pos_tags'].apply(pos_distribution)

print("=== context vs question 품사 분포 비교 ===\n")

for field in ['context', 'question']:
    subset = df_samples[df_samples['field'] == field]
    # 전체 품사를 합산
    all_tags = []
    for tags in subset['pos_tags']:
        all_tags.extend(pos for _, pos in tags)

    dist = Counter(all_tags)
    total = sum(dist.values())
    print(f"[{field}] 품사 분포 (전체 {total}개 형태소)")
    for pos, cnt in dist.most_common():
        bar = '█' * int(cnt / total * 40)
        print(f"  {pos:<12}: {cnt:>4}개 ({cnt/total*100:5.1f}%) {bar}")
    print()

In [ ]:
# ── 최종 요약 테이블 ───────────────────────────────────────────────
print("=== POS Tagging 요약 테이블 ===\n")

summary_pos = df_samples[[
    'category', 'condition', 'field',
    'text', 'pos_tags', 'nouns', 'verbs'
]].copy()

summary_pos['n_tokens']    = summary_pos['pos_tags'].apply(len)
summary_pos['n_nouns']     = summary_pos['nouns'].apply(len)
summary_pos['n_verbs']     = summary_pos['verbs'].apply(len)
summary_pos['noun_ratio']  = (
    summary_pos['n_nouns'] / summary_pos['n_tokens'].replace(0, 1) * 100
).round(1)

display(summary_pos[[
    'category', 'condition', 'field',
    'n_tokens', 'n_nouns', 'n_verbs', 'noun_ratio'
]])

print("\n=== 전체 평균 ===")
print(summary_pos[['n_tokens', 'n_nouns', 'n_verbs', 'noun_ratio']].mean().round(2))

# 4. short interpretation

KoBBQ는 편향 탐지 데이터셋으로, POS Tagging을 통해 편향 대상이 되는 핵심 명사(인물, 집단)를 카테고리별로 추출할 수 있었다. 또한 context는 명사 비율이 높고, question은 동사 접두사·형용사 비율이 상대적으로 높다는 구조적 차이를 확인할 수 있었다. 이는 KoBBQ의 질문이 '누가 ~하지 않았습니까?'처럼 부정 표현을 포함한 구조로 설계된 경우가 많기 때문이다. 이러한 부정 질문 패턴은 특정 집단에 대한 부정적 편향을 유도하는 KoBBQ의 설계 의도와도 연결된다.